<a href="https://colab.research.google.com/github/gitcnk/Time_Series_Forecasting/blob/main/jena_climate_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN Time Series Forecasting on Jena Climate Dataset

This notebook builds a SimpleRNN model to predict temperature using all 14 variables from the Jena Climate dataset.

**README:** Please use a GPU runtime for fast execution.  It takes about 100 seconds per epoch with batch size of 256.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

2026-04-20 13:15:25.652782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776690925.831788      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776690925.882454      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776690926.293587      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776690926.293627      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776690926.293630      23 computation_placer.cc:177] computation placer alr

## 2. Load and Explore the Dataset

In [2]:
!wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
!unzip jena_climate_2009_2016.csv.zip

import os
fname = os.path.join("jena_climate_2009_2016.csv")


--2026-04-20 13:15:49--  https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.142.160, 52.216.218.32, 52.217.192.64, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.142.160|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13565642 (13M) [application/zip]
Saving to: ‘jena_climate_2009_2016.csv.zip’

jena_climate_2009_2 100%[===================>]  12.94M  49.8MB/s    in 0.3s    

2026-04-20 13:15:49 (49.8 MB/s) - ‘jena_climate_2009_2016.csv.zip’ saved [13565642/13565642]

Archive:  jena_climate_2009_2016.csv.zip
  inflating: jena_climate_2009_2016.csv  
  inflating: __MACOSX/._jena_climate_2009_2016.csv  


In [3]:
# Load the Jena Climate dataset
df = pd.read_csv(fname)

# Display dataset info
df.shape

df.head()


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


## Choose the Predictors for the model

In [4]:
predictor_indices = [1, 2, 4, 5]  # Pressure, Temperature(C), Dew, Rel.Humidity
data = df.iloc[:, predictor_indices].values

data[0:4, :]

array([[996.52,  -8.02,  -8.9 ,  93.3 ],
       [996.57,  -8.41,  -9.28,  93.4 ],
       [996.53,  -8.51,  -9.31,  93.9 ],
       [996.51,  -8.31,  -9.07,  94.2 ]])

In [5]:
# Normalize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

## 4. Create Time Series Datasets

Use `keras.utils.timeseries_dataset_from_array` to create batched time series datasets.

In [6]:
# Hyperparameters
seq_length = 120  # Use 100 timesteps to predict the next value
batch_size = 1024
train_split = int(len(data_scaled) * 0.8)

# Split data before creating datasets
train_data = data_scaled[:train_split]
test_data = data_scaled[train_split:]

In [7]:
# Create training dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    data=train_data,
    targets=train_data[:, 1],  # Predict temperature (second column)
    sequence_length=seq_length,
    sampling_rate=1,
    batch_size=batch_size,
    shuffle=True,
    seed=42
)

I0000 00:00:1776690952.301623      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [8]:
# Create validation dataset (from last 20% of training data)
val_split = int(len(train_data) * 0.8)
val_data = train_data[val_split:]
val_targets = train_data[val_split:, 1]

val_dataset = keras.utils.timeseries_dataset_from_array(
    data=train_data,
    targets=train_data[:, 1],
    sequence_length=seq_length,
    sampling_rate=1,
    batch_size=batch_size,
    shuffle=False
)

In [9]:
# Create test dataset
test_dataset = keras.utils.timeseries_dataset_from_array(
    data=test_data,
    targets=test_data[:, 1],  # Predict temperature
    sequence_length=seq_length,
    sampling_rate=1,
    batch_size=batch_size,
    shuffle=False
)

train_data.shape
test_data.shape
batch_size

1024

## 5. Build the RNN Model

In [10]:
# Number of predictors
p = 4

# Build RNN model
model = keras.Sequential([
    layers.SimpleRNN(256, activation='relu', input_shape=(seq_length, p), return_sequences=True),
    layers.Dropout(0.2),
    layers.SimpleRNN(128, activation='relu', return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
# Simple NN

p = 4

model = keras.Sequential([
    # Input: (batch_size, seq_length=100, num_predictors=p)
    layers.Dense(256,
                 activation='relu',
                 input_shape=(seq_length, p)),
    layers.Dropout(0.3),

    # Second Dense layer
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Third Dense layer
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    # Fourth Dense layer
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),

    # Flatten: (batch_size, seq_length, 32) -> (batch_size, seq_length*32)
    layers.Flatten(),

    # Fifth Dense layer
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.1),

    # Output layer
    layers.Dense(1)
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 120, 256)       │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 120, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 120, 128)       │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 120, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 120, 64)        │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 120, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 120, 32)        │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 120, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │        61,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,985 (414.00 KB)

 Trainable params: 105,985 (414.00 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train the Model

In [13]:
# Add this BEFORE model.fit()

import subprocess
import threading
import time

def gpu_monitor():
    while True:
        try:
            subprocess.run(['nvidia-smi'])
            time.sleep(10)  # Refresh every 10 seconds
        except:
            break

# Start background thread
t = threading.Thread(target=gpu_monitor, daemon=True)
#t.start()

# Now run your training - GPU monitor prints in background
#history = model.fit(
    #train_dataset,
    #epochs=30,
    #validation_data=val_dataset,
    #verbose=1
#)

In [14]:

# Train the model
#history = model.fit(
    #train_dataset,
    #epochs=10,
    #validation_data=val_dataset,
    #verbose=1
#)

## 7. Evaluate on Test Set

In [15]:
# Evaluate on test set
#test_loss, test_mae = model.evaluate(test_dataset)


## 8. Make Predictions and Calculate Metrics

In [16]:
# Make predictions on test set
#y_pred_test = model.predict(test_dataset, verbose=0)

# Get actual test targets
#y_test = np.concatenate([y for x, y in test_dataset], axis=0)

In [17]:
# Inverse transform to original scale (only for temperature - column 0)
# Create dummy arrays with the right shape for inverse transform
#y_test_dummy = np.zeros((len(y_test), 14))
#y_test_dummy[:, 1] = y_test.flatten()
#y_test_original = scaler.inverse_transform(y_test_dummy)[:, 1]

#y_pred_test_dummy = np.zeros((len(y_pred_test), 14))
#y_pred_test_dummy[:, 1] = y_pred_test.flatten()
#y_pred_test_original = scaler.inverse_transform(y_pred_test_dummy)[:, 1]

In [18]:
# Calculate RMSE
#test_rmse = np.sqrt(np.mean((y_pred_test_original - y_test_original) ** 2))


## 9. Visualize Results

In [19]:
# Plotting
#fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot training history
#axes[0].plot(history.history['loss'], label='Training Loss')
#axes[0].plot(history.history['val_loss'], label='Validation Loss')
#axes[0].set_title('Model Loss During Training')
#axes[0].set_xlabel('Epoch')
#axes[0].set_ylabel('Loss')
#axes[0].legend()
#axes[0].grid(True, alpha=0.3)

# Plot predictions vs actual
#test_indices = range(len(y_pred_test_original))
#axes[1].plot(test_indices, y_test_original, label='Actual', linewidth=2)
#axes[1].plot(test_indices, y_pred_test_original, label='Predicted', alpha=0.8)
#axes[1].set_title('Temperature Predictions vs Actual (Test Set)')
#axes[1].set_xlabel('Time Step')
#axes[1].set_ylabel('Temperature (°C)')
#axes[1].legend()
#axes[1].grid(True, alpha=0.3)

#plt.tight_layout()
#plt.show()

In [20]:
#test_indices = range(len(y_pred_test_original))
#plt.plot(test_indices, y_test_original, label='Actual', linewidth=2)
#plt.plot(test_indices, y_pred_test_original, label='Predicted', alpha=0.8)
#plt.title('Temperature Predictions vs Actual (Test Set)')
#plt.xlabel('Time Step')
#plt.ylabel('Temperature (°C)')
#plt.show()